Group XX | Members: [Student 1], [Student 2] | Question: Q1 | Stage: 5: feature engineering and selection

How this works: quantify engineered feature usefulness with correlation, mutual information, and train-only random forest importance.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / 'src').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd, numpy as np
from pathlib import Path
from sklearn.feature_selection import mutual_info_regression
from src.config import PROCESSED_DIR, DOCS_DIR, TARGET_COL
from src.features import remove_high_corr, rf_importance, select_features_numeric

DOCS_DIR.mkdir(parents=True, exist_ok=True)
train = pd.read_csv(PROCESSED_DIR / 'train_clean_unscaled.csv')
val = pd.read_csv(PROCESSED_DIR / 'validation_clean_unscaled.csv')
test = pd.read_csv(PROCESSED_DIR / 'test_clean_unscaled.csv')

feature_candidates = [c for c in train.columns if c not in ['farm_id', TARGET_COL, 'harvest_weight_kg', 'planting_date', 'planting_dt']]
X_train = train[feature_candidates].copy()
y_train = train[TARGET_COL]

# numeric-only analysis for MI and selection summary
X_num = X_train.select_dtypes(include=['number']).copy()
X_num2, dropped_corr = remove_high_corr(X_num, threshold=0.9)
kept_var, selected_num = select_features_numeric(X_num2, y_train, k=min(15, max(1, X_num2.shape[1])))

mi = mutual_info_regression(X_num2.fillna(X_num2.median()), y_train, random_state=42)
mi_df = pd.DataFrame({'feature': X_num2.columns, 'mutual_info': mi}).sort_values('mutual_info', ascending=False)
rf_df = rf_importance(X_train, y_train)

feature_rows = []
def add_row(f, how, evidence, q):
    feature_rows.append({'feature':f,'created_or_keep_remove_reason':how,'evidence':evidence,'linked_question':q})

core_new = ['fertilizer_kg_per_ha','planting_month','planting_quarter','days_from_season_start','is_improved_seed','seed_rank','altitude_band','temp_deviation_from_optimum','rain_per_temp','rainfall_log','fertilizer_log1p','farm_size_log','irrigated_flag','season_x_irrigation']
for f in core_new:
    if f in train.columns:
        add_row(f, 'created', 'Domain-informed feature engineering from Stage 5', 'B/C/D/E')

for f in dropped_corr:
    add_row(f, 'removed', 'Highly correlated pair |r|>0.9', 'A/B/C')

for f in selected_num:
    add_row(f, 'kept', 'Selected by variance+f_regression on train set', 'A/B/C')

fj = pd.DataFrame(feature_rows).drop_duplicates()
fj.to_csv(DOCS_DIR / 'feature_justification.csv', index=False)
(DOCS_DIR / 'feature_justification.md').write_text('# Feature Justification\n\n' + fj.to_string(index=False) + '\n\n## Top Mutual Information\n\n' + mi_df.head(15).to_string(index=False) + '\n\n## RandomForest Importance (train only)\n\n' + rf_df.head(15).to_string(index=False))

# save selected feature list
selected_all = sorted(set(selected_num + [c for c in feature_candidates if c not in X_num.columns]))
(Path(PROCESSED_DIR / 'selected_features.txt')).write_text('\n'.join(selected_all))
print('Engineered feature count in train:', len([f for f in core_new if f in train.columns]))
print('Selected numeric features:', len(selected_num))
print('Dropped correlated numeric features:', dropped_corr)


Engineered feature count in train: 14
Selected numeric features: 15
Dropped correlated numeric features: ['planting_quarter', 'rain_per_temp', 'rainfall_log']
